# Flute → Indian Classical Notes: Two-Pipeline Comparison

Based on the discussion around key detection and tonic identification, this notebook tests **two end-to-end pipelines**:

---

## Pipeline A — Direct Conversion
```
Flute Audio  ──►  Pitch Tracker  ──►  Sa Detection  ──►  Indian Swaras
                 (PYIN / CREPE /                         (Sa Re Ga Ma...)
                  HPS / Basic-Pitch)
```

## Pipeline B — Western Intermediate
```
Flute Audio  ──►  Western Transcription  ──►  Sa/Tonic Detection  ──►  Indian Swaras
                 (MT3 / Basic-Pitch /         (Chromagram PCP /
                  CREPE → MIDI)               Krumhansl-Schmuckler /
                                              Low-Freq Isolation)
```

---
### Sa Detection Methods (from the discussion)
| Method | Idea |
|--------|------|
| **PCP Peak Pair** | Sa & Pa are played most; find the two most common notes a perfect fifth apart |
| **Krumhansl-Schmuckler** | Slide a statistical scale template over the chroma; best-fit position = Sa |
| **Low-Freq Isolation** | Low-pass filter → only drone/bass remains; its dominant pitch is Sa |

## 0. Install Dependencies

In [ ]:
# Core audio + ML libraries
!pip install librosa soundfile numpy matplotlib scipy crepe basic-pitch --quiet

# MT3 (Google Magenta multi-task transcription) — requires JAX
# Uncomment the block below if you want to run MT3 locally.
# It needs ~4 GB RAM and works best on GPU/TPU.

# !pip install jax[cuda12] -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html --quiet
# !pip install git+https://github.com/magenta/mt3.git --quiet
# !gsutil -q -m cp -r gs://mt3/checkpoints .

print('Dependencies ready.')

## 1. Shared Setup

In [ ]:
import numpy as np
import librosa
import librosa.display
import soundfile as sf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.signal import butter, filtfilt
from scipy.ndimage import uniform_filter1d
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['font.size'] = 11

# ── Indian Classical Note Definitions ─────────────────────────────────────────
SWARAS_12TET = {
    0:  ('Sa',   'Shadja',   'shuddha'),
    1:  ('Re♭',  'Rishabh',  'komal'),
    2:  ('Re',   'Rishabh',  'shuddha'),
    3:  ('Ga♭',  'Gandhar',  'komal'),
    4:  ('Ga',   'Gandhar',  'shuddha'),
    5:  ('Ma',   'Madhyam',  'shuddha'),
    6:  ('Ma#',  'Madhyam',  'tivra'),
    7:  ('Pa',   'Pancham',  'shuddha'),
    8:  ('Dha♭', 'Dhaivat',  'komal'),
    9:  ('Dha',  'Dhaivat',  'shuddha'),
    10: ('Ni♭',  'Nishad',   'komal'),
    11: ('Ni',   'Nishad',   'shuddha'),
}

SWARA_COLORS = {
    0:'#FF4444', 1:'#FF8C44', 2:'#FFC044', 3:'#F0E040',
    4:'#88CC44', 5:'#44CC88', 6:'#44CCCC', 7:'#4488FF',
    8:'#8844FF', 9:'#CC44FF',10:'#FF44CC',11:'#FF4488',
}

WESTERN_NOTES = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']

def hz_to_midi(freq_hz):
    return 69 + 12 * np.log2(freq_hz / 440.0)

def midi_to_hz(midi):
    return 440.0 * (2 ** ((midi - 69) / 12))

def semitone_to_swara(semitone_offset):
    """Semitone offset from Sa (0–11) → swara dict."""
    st = int(semitone_offset) % 12
    short, full, variant = SWARAS_12TET[st]
    return {'semitone': st, 'short_name': short, 'full_name': full, 'variant': variant}

def hz_to_swara(freq_hz, sa_hz):
    """Frequency → swara given a tonic Sa frequency."""
    if not freq_hz or np.isnan(freq_hz) or freq_hz <= 0:
        return None
    semitones_from_sa = 12 * np.log2(freq_hz / sa_hz)
    octave = int(np.floor(semitones_from_sa / 12))
    semitone_in_oct = semitones_from_sa - octave * 12
    nearest = int(round(semitone_in_oct)) % 12
    cents_dev = (semitone_in_oct - round(semitone_in_oct)) * 100
    short, full, variant = SWARAS_12TET[nearest]
    return {
        'semitone': nearest, 'short_name': short, 'full_name': full,
        'variant': variant, 'octave': octave,
        'cents_deviation': round(cents_dev, 1), 'freq_hz': round(freq_hz, 2),
    }

def collapse_runs(seq, key='short_name'):
    """Collapse frame-level sequence into run-length events."""
    runs, current, start_t = [], None, None
    for item in seq:
        label = item.get(key) if item else None
        t = item.get('time') if item else None
        if label != current:
            if current is not None:
                runs.append({key: current, 'start': start_t, 'end': t})
            current, start_t = label, t
    if current is not None and seq:
        runs.append({key: current, 'start': start_t, 'end': seq[-1].get('time')})
    return runs

print('Setup complete.')

## 2. Load Audio

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
AUDIO_FILE = 'your_flute_piece.wav'   # <-- set your file path here
SR         = 22050

# ── Load ──────────────────────────────────────────────────────────────────────
y, sr = librosa.load(AUDIO_FILE, sr=SR, mono=True)
duration = librosa.get_duration(y=y, sr=sr)
print(f'Loaded  : {AUDIO_FILE}')
print(f'Duration: {duration:.2f}s  |  SR: {sr} Hz')

plt.figure(figsize=(14, 2.5))
librosa.display.waveshow(y, sr=sr, alpha=0.7, color='steelblue')
plt.title('Waveform')
plt.tight_layout(); plt.show()

In [ ]:
# ── Synthetic test audio (Sa–Re–Ga–Ma–Pa–Dha–Ni–Sa in G) ──────────────────────
# Uses G4 (392 Hz) as Sa.  Uncomment the last 3 lines to use this instead.

SA_SYNTH_HZ = 392.0   # G4

def make_synth_flute(sa_hz=392.0, sr=22050, note_dur=0.6):
    # Ascending shuddha saptak + some komal notes for interest
    semitones = [0, 0, 2, 4, 5, 7, 7, 9, 11, 12, 11, 9, 7, 5, 4, 2, 0]
    audio = []
    for st in semitones:
        freq = sa_hz * (2 ** (st / 12))
        t    = np.linspace(0, note_dur, int(sr * note_dur), endpoint=False)
        wave = (0.55 * np.sin(2*np.pi*freq*t)
              + 0.25 * np.sin(2*np.pi*2*freq*t)
              + 0.12 * np.sin(2*np.pi*3*freq*t)
              + 0.05 * np.sin(2*np.pi*4*freq*t))
        env  = np.ones_like(t)
        env[:int(0.04*sr)] = np.linspace(0, 1, int(0.04*sr))
        env[-int(0.08*sr):] = np.linspace(1, 0, int(0.08*sr))
        audio.append(wave * env)
    return np.concatenate(audio)

y_synth = make_synth_flute(sa_hz=SA_SYNTH_HZ)
sf.write('synth_flute_G.wav', y_synth, SR)
print('Synthetic flute (Sa=G4=392Hz) saved → synth_flute_G.wav')

# Uncomment to use:
# y, sr = y_synth, SR
# AUDIO_FILE = 'synth_flute_G.wav'

---
# PIPELINE B — Western Intermediate

We do this first because it produces the chromagram and Western MIDI that Pipeline A also reuses.

## B-Step 1: Transcribe to Western Notes

Three methods: **MT3**, **Basic-Pitch**, **CREPE → MIDI**

### B1a — MT3 (Google Magenta)

MT3 is a T5-based seq2seq model fine-tuned on multi-track MIDI. It reads audio spectrograms and outputs MIDI token sequences. It's the most accurate option for complex music.

In [ ]:
# ── MT3 Transcription ─────────────────────────────────────────────────────────
# Requires: JAX, T5X, and the MT3 checkpoint downloaded to ./checkpoints/mt3/
# Setup:
#   pip install jax[cuda12] -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
#   pip install git+https://github.com/magenta/mt3.git
#   gsutil -q -m cp -r gs://mt3/checkpoints .

MT3_AVAILABLE = False   # flip to True after installing above

midi_data_mt3   = None
note_events_mt3 = []

if MT3_AVAILABLE:
    import functools
    import gin
    import jax
    import note_seq
    import seqio
    import t5
    import t5x
    import mt3

    CHECKPOINT_PATH = './checkpoints/mt3/'   # path to downloaded checkpoint

    # Build the inference model
    gin.parse_config_file(f'{CHECKPOINT_PATH}/config.gin')
    model = mt3.colab_utils.load_model(checkpoint_path=CHECKPOINT_PATH)

    # Run inference
    ns = mt3.colab_utils.transcribe_audio(
        model=model,
        audio_filename=AUDIO_FILE,
        sample_rate=sr,
    )

    # Convert NoteSequence → list of (start, end, pitch, velocity)
    note_events_mt3 = [
        (n.start_time, n.end_time, n.pitch, n.velocity)
        for n in ns.notes
    ]

    print(f'MT3: {len(note_events_mt3)} notes detected')
    for start, end, pitch, vel in note_events_mt3[:10]:
        note_name = librosa.midi_to_note(pitch)
        print(f'  {start:.3f}s – {end:.3f}s  MIDI={pitch}  ({note_name})')
else:
    print('MT3 not installed. Set MT3_AVAILABLE=True after setup.')
    print('Falling back to Basic-Pitch and CREPE for Western transcription.')

### B1b — Basic-Pitch (Spotify)

In [ ]:
from basic_pitch.inference import predict
from basic_pitch import ICASSP_2022_MODEL_PATH

_, midi_bp, note_events_bp = predict(AUDIO_FILE)

# note_events_bp: list of (start, end, pitch_midi, amplitude, pitch_bends)
note_events_bp_clean = [
    (start, end, pitch, amp)
    for start, end, pitch, amp, _ in note_events_bp
]

print(f'Basic-Pitch: {len(note_events_bp_clean)} notes')
for start, end, pitch, amp in note_events_bp_clean[:10]:
    note_name = librosa.midi_to_note(pitch)
    print(f'  {start:.3f}s – {end:.3f}s  MIDI={pitch}  ({note_name})  amp={amp:.2f}')

### B1c — CREPE → MIDI

In [ ]:
import crepe

y_16k = librosa.resample(y, orig_sr=sr, target_sr=16000)
time_crepe, freq_crepe, conf_crepe, _ = crepe.predict(
    y_16k, 16000, model='small', step_size=10, viterbi=True, verbose=0
)

CONF_THRESH = 0.5
f0_crepe_voiced = np.where(conf_crepe >= CONF_THRESH, freq_crepe, np.nan)

# Convert to MIDI-like note events (group voiced frames into notes)
def pitch_frames_to_note_events(times, freqs, confs, conf_thresh=0.5, min_dur=0.05):
    """Group consecutive voiced frames into note events."""
    events, in_note, note_freqs, start_t = [], False, [], None
    for t, f, c in zip(times, freqs, confs):
        voiced = c >= conf_thresh and not np.isnan(f) and f > 0
        if voiced and not in_note:
            in_note, start_t, note_freqs = True, t, [f]
        elif voiced and in_note:
            note_freqs.append(f)
        elif not voiced and in_note:
            dur = t - start_t
            if dur >= min_dur:
                median_freq = np.median(note_freqs)
                midi = round(hz_to_midi(median_freq))
                events.append((start_t, t, int(midi), median_freq))
            in_note, note_freqs = False, []
    return events

note_events_crepe = pitch_frames_to_note_events(time_crepe, freq_crepe, conf_crepe, CONF_THRESH)

print(f'CREPE → MIDI: {len(note_events_crepe)} note events')
for start, end, midi, freq in note_events_crepe[:10]:
    note_name = librosa.midi_to_note(midi)
    print(f'  {start:.3f}s – {end:.3f}s  MIDI={midi}  ({note_name})  {freq:.1f}Hz')

---
## B-Step 2: Sa / Tonic Detection

Three methods from the discussion:
1. **PCP Peak Pair** — Sa & Pa are most common; find the pair a perfect fifth apart
2. **Krumhansl-Schmuckler template matching**
3. **Low-frequency isolation** (drone / tanpura bass)

In [ ]:
# ── Build Chromagram (shared by all three methods) ────────────────────────────
# CQT-based chromagram: better frequency resolution than STFT chroma for music
hop_length = 512
chroma_cqt = librosa.feature.chroma_cqt(y=y, sr=sr, hop_length=hop_length, bins_per_octave=36)

# Aggregate: mean energy per pitch class over entire piece
pcp = chroma_cqt.mean(axis=1)   # Pitch Class Profile, shape=(12,)
pcp /= pcp.sum()                 # normalise to probabilities

fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))

librosa.display.specshow(chroma_cqt, y_axis='chroma', x_axis='time',
                         hop_length=hop_length, ax=axes[0])
axes[0].set_title('Chromagram (CQT)')

axes[1].bar(WESTERN_NOTES, pcp, color='steelblue', edgecolor='white')
axes[1].set_ylabel('Pitch Class Probability')
axes[1].set_title('Pitch Class Profile (PCP) — the "statistical bar chart"')

plt.tight_layout(); plt.show()
print('PCP:', {n: round(float(p), 3) for n, p in zip(WESTERN_NOTES, pcp)})

### Sa Detection Method 1 — PCP Peak Pair (Sa + Pa = Perfect Fifth)

In [ ]:
def detect_sa_pcp_pair(pcp, notes=WESTERN_NOTES):
    """
    From the discussion:
    'Sa and Pa are mathematically played the most often.
     Find the two biggest peaks that are a perfect fifth (7 semitones) apart.'

    Strategy:
    - Score each pitch class as Sa candidate by:
        pcp[candidate] + pcp[(candidate+7)%12]   (Sa + Pa)
    - The candidate with highest combined score is Sa.
    """
    scores = np.array([
        pcp[i] + pcp[(i + 7) % 12]   # Sa energy + Pa energy
        for i in range(12)
    ])

    best_sa_idx = int(np.argmax(scores))
    best_pa_idx = (best_sa_idx + 7) % 12

    print(f'[PCP Pair]  Sa candidate : {notes[best_sa_idx]}  (score={scores[best_sa_idx]:.4f})')
    print(f'            Pa candidate : {notes[best_pa_idx]}')

    # Visualise scores
    fig, ax = plt.subplots(figsize=(10, 3))
    bars = ax.bar(notes, scores, color='coral', edgecolor='white')
    bars[best_sa_idx].set_color('#FF4444')
    bars[best_sa_idx].set_edgecolor('black')
    bars[best_sa_idx].set_linewidth(2)
    ax.set_title(f'PCP Pair Scores (Sa+Pa combined) — Best Sa = {notes[best_sa_idx]}')
    ax.set_ylabel('Sa+Pa score')
    plt.tight_layout(); plt.show()

    return best_sa_idx, notes[best_sa_idx]


sa_idx_pcp, sa_name_pcp = detect_sa_pcp_pair(pcp)
sa_hz_pcp = librosa.note_to_hz(sa_name_pcp + '4')   # assume octave 4 as reference
print(f'Sa (PCP Pair): {sa_name_pcp}  ≈  {sa_hz_pcp:.2f} Hz')

### Sa Detection Method 2 — Krumhansl-Schmuckler Template Matching

In [ ]:
def detect_sa_krumhansl(pcp, notes=WESTERN_NOTES):
    """
    From the discussion:
    'Advanced machines use the Krumhansl-Schmuckler algorithm.
     The machine holds a mathematical template of what a scale looks like
     and slides it across the data. Once it locks in, the machine found home base.'

    The K-S key profiles (major and minor) give expected PCP weights for each
    scale degree.  We compute the Pearson correlation between the observed PCP
    and the template for every rotation (= every possible root note).
    """
    # Krumhansl-Kessler (1982) key profiles
    # Major: strong weights on 1, 3, 5; weaker on passing tones
    KS_MAJOR = np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09,
                          2.52, 5.19, 2.39, 3.66, 2.29, 2.88])
    # Minor (natural minor profile)
    KS_MINOR = np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53,
                          2.54, 4.75, 3.98, 2.69, 3.34, 3.17])

    # Also include a Bhairav (Indian) template — semitones: 0,1,4,5,7,8,11
    KS_BHAIRAV = np.zeros(12)
    for st in [0, 1, 4, 5, 7, 8, 11]:
        KS_BHAIRAV[st] = 1.0

    # Include Yaman (Kalyan): semitones 0,2,4,6,7,9,11
    KS_YAMAN = np.zeros(12)
    for st in [0, 2, 4, 6, 7, 9, 11]:
        KS_YAMAN[st] = 1.0

    templates = {
        'Major':  KS_MAJOR,
        'Minor':  KS_MINOR,
        'Bhairav': KS_BHAIRAV,
        'Yaman':  KS_YAMAN,
    }

    results = {}
    for tname, template in templates.items():
        corrs = []
        for rotation in range(12):
            rotated = np.roll(template, rotation)
            # Pearson correlation
            corr = np.corrcoef(pcp, rotated)[0, 1]
            corrs.append(corr)
        best_rot = int(np.argmax(corrs))
        results[tname] = {'sa_idx': best_rot, 'sa_name': notes[best_rot],
                          'correlation': corrs[best_rot], 'all_corrs': corrs}

    # Print results
    print('[Krumhansl-Schmuckler] Template matching results:')
    for tname, r in results.items():
        print(f'  {tname:<10}  Sa={r["sa_name"]}  corr={r["correlation"]:.4f}')

    # Visualise correlation curves
    fig, axes = plt.subplots(1, len(templates), figsize=(14, 3.5), sharey=True)
    for ax, (tname, r) in zip(axes, results.items()):
        bars = ax.bar(notes, r['all_corrs'], color='mediumpurple', edgecolor='white')
        bars[r['sa_idx']].set_color('#FF4444')
        bars[r['sa_idx']].set_edgecolor('black')
        bars[r['sa_idx']].set_linewidth(2)
        ax.set_title(f'{tname} template\nSa={r["sa_name"]} ({r["correlation"]:.3f})')
        ax.set_ylabel('Pearson correlation' if ax == axes[0] else '')
        ax.tick_params(axis='x', rotation=45)
    plt.suptitle('Krumhansl-Schmuckler: Template Correlation per Root Candidate', y=1.02)
    plt.tight_layout(); plt.show()

    # Best overall: highest correlation across all templates
    best = max(results.items(), key=lambda x: x[1]['correlation'])
    print(f'\nBest overall: {best[0]} — Sa={best[1]["sa_name"]}  corr={best[1]["correlation"]:.4f}')
    return best[1]['sa_idx'], best[1]['sa_name'], best[0]


sa_idx_ks, sa_name_ks, best_template = detect_sa_krumhansl(pcp)
sa_hz_ks = librosa.note_to_hz(sa_name_ks + '4')
print(f'Sa (Krumhansl-Schmuckler): {sa_name_ks}  ≈  {sa_hz_ks:.2f} Hz')

### Sa Detection Method 3 — Low-Frequency Isolation (Drone / Tanpura Bass)

In [ ]:
def detect_sa_low_freq(y, sr, cutoff_hz=400.0, notes=WESTERN_NOTES):
    """
    From the discussion:
    'The machine uses a Low-Pass Filter. It intentionally deletes all high
     frequencies and only analyzes the lowest rumbling frequencies.
     Bass instruments and drones almost exclusively play Sa and Pa,
     so isolating the bass makes it incredibly easy to find the root note.'
    """
    # Low-pass Butterworth filter
    nyq = sr / 2
    b, a = butter(4, cutoff_hz / nyq, btype='low')
    y_low = filtfilt(b, a, y)

    # Chromagram of the low-passed signal
    chroma_low = librosa.feature.chroma_cqt(y=y_low, sr=sr, hop_length=512,
                                             fmin=librosa.note_to_hz('C1'))
    pcp_low = chroma_low.mean(axis=1)
    pcp_low /= pcp_low.sum()

    best_sa_idx = int(np.argmax(pcp_low))

    fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))

    # Spectrogram comparison: full vs low-passed
    D_full = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    D_low  = librosa.amplitude_to_db(np.abs(librosa.stft(y_low)), ref=np.max)

    librosa.display.specshow(D_low, sr=sr, x_axis='time', y_axis='log', ax=axes[0])
    axes[0].axhline(cutoff_hz, color='red', linestyle='--', linewidth=1.5, label=f'cutoff={cutoff_hz}Hz')
    axes[0].set_title('Low-Pass Filtered Spectrogram')
    axes[0].legend()

    bars = axes[1].bar(notes, pcp_low, color='mediumseagreen', edgecolor='white')
    bars[best_sa_idx].set_color('#FF4444')
    bars[best_sa_idx].set_edgecolor('black')
    bars[best_sa_idx].set_linewidth(2)
    axes[1].set_title(f'PCP (Bass Only) — Sa = {notes[best_sa_idx]}')
    axes[1].set_ylabel('Probability')

    plt.tight_layout(); plt.show()
    print(f'[Low-Freq Isolation]  Sa = {notes[best_sa_idx]}  (cutoff={cutoff_hz}Hz)')
    return best_sa_idx, notes[best_sa_idx]


sa_idx_lf, sa_name_lf = detect_sa_low_freq(y, sr, cutoff_hz=400.0)
sa_hz_lf = librosa.note_to_hz(sa_name_lf + '4')
print(f'Sa (Low-Freq):  {sa_name_lf}  ≈  {sa_hz_lf:.2f} Hz')

### Sa Detection — Comparison & Consensus

In [ ]:
# Vote across all three methods
sa_votes = [sa_name_pcp, sa_name_ks, sa_name_lf]
sa_counts = Counter(sa_votes)
sa_consensus, sa_consensus_votes = sa_counts.most_common(1)[0]

print('=== Sa Detection Summary ===')
print(f'  PCP Pair            : {sa_name_pcp}')
print(f'  Krumhansl-Schmuckler: {sa_name_ks}  (best template: {best_template})')
print(f'  Low-Freq Isolation  : {sa_name_lf}')
print(f'  CONSENSUS Sa        : {sa_consensus}  ({sa_consensus_votes}/3 methods agree)')

# Use consensus for the rest of Pipeline B
SA_HZ = librosa.note_to_hz(sa_consensus + '4')
print(f'\nUsing Sa = {sa_consensus}  ({SA_HZ:.2f} Hz) for Western → Indian mapping.')

## B-Step 3: Map Western Notes → Indian Swaras

Now that we know Sa, every Western MIDI pitch is just a semitone offset from Sa.

In [ ]:
def western_notes_to_swaras(note_events, sa_hz, source_label=''):
    """
    Convert a list of (start, end, midi_pitch, ...) note events
    into timestamped swara events.
    """
    swara_events = []
    for event in note_events:
        start, end, midi_pitch = event[0], event[1], event[2]
        freq = midi_to_hz(midi_pitch)
        swara = hz_to_swara(freq, sa_hz=sa_hz)
        if swara:
            swara['start'] = start
            swara['end']   = end
            swara['midi']  = midi_pitch
            swara_events.append(swara)
    return swara_events


# Apply to all three Western transcription methods
swaras_from_bp    = western_notes_to_swaras(note_events_bp_clean, SA_HZ, 'BasicPitch')
swaras_from_crepe = western_notes_to_swaras(note_events_crepe,    SA_HZ, 'CREPE')

if note_events_mt3:
    swaras_from_mt3 = western_notes_to_swaras(note_events_mt3, SA_HZ, 'MT3')
else:
    swaras_from_mt3 = []   # MT3 not available

# Print transcript
def print_swara_transcript(swara_events, label, max_events=20):
    print(f'\n{label} — first {min(max_events, len(swara_events))} swaras:')
    print('  Time              MIDI  Freq    Swara  Octave  Cents')
    for s in swara_events[:max_events]:
        print(f"  {s['start']:.2f}–{s['end']:.2f}s  "
              f"  {s['midi']}   {s['freq_hz']:.0f}Hz  "
              f"  {s['short_name']:<5}  {s['octave']:+d}      {s['cents_deviation']:+.0f}c")

print_swara_transcript(swaras_from_bp,    'Basic-Pitch → Swaras')
print_swara_transcript(swaras_from_crepe, 'CREPE → Swaras')
if swaras_from_mt3:
    print_swara_transcript(swaras_from_mt3, 'MT3 → Swaras')

In [ ]:
# Piano-roll style visualisation for Pipeline B
sources = []
if swaras_from_bp:    sources.append(('Basic-Pitch → Swaras', swaras_from_bp,    'tomato'))
if swaras_from_crepe: sources.append(('CREPE → Swaras',       swaras_from_crepe, 'steelblue'))
if swaras_from_mt3:   sources.append(('MT3 → Swaras',         swaras_from_mt3,   'gold'))

n = len(sources)
fig, axes = plt.subplots(n, 1, figsize=(14, 4 * n), sharex=True)
if n == 1: axes = [axes]

for ax, (label, events, color) in zip(axes, sources):
    for s in events:
        ax.barh(y=s['semitone'], width=s['end']-s['start'], left=s['start'],
                height=0.75, color=SWARA_COLORS[s['semitone']], alpha=0.85)
        if (s['end'] - s['start']) > 0.15:
            ax.text(s['start'] + (s['end']-s['start'])/2, s['semitone'],
                    s['short_name'], ha='center', va='center',
                    fontsize=7, color='white', fontweight='bold')
    ax.set_yticks(range(12))
    ax.set_yticklabels([SWARAS_12TET[i][0] for i in range(12)])
    ax.set_title(f'{label}  (Sa={sa_consensus})')
    ax.grid(axis='x', alpha=0.3)

axes[-1].set_xlabel('Time (s)')
plt.suptitle('Pipeline B — Western Intermediate → Indian Swaras', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

---
# PIPELINE A — Direct Conversion

Skip the Western step entirely. Pitch track the audio directly and map to swaras using the Sa detected in B-Step 2.

## A-Step 1: PYIN

In [ ]:
f0_pyin, voiced_pyin, _ = librosa.pyin(
    y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'),
    sr=sr, frame_length=2048, hop_length=512
)
times_pyin = librosa.times_like(f0_pyin, sr=sr, hop_length=512)
f0_pyin_v  = np.where(voiced_pyin, f0_pyin, np.nan)

seq_pyin = [
    {**hz_to_swara(float(f), SA_HZ), 'time': float(t)} if not np.isnan(float(f)) else {'time': float(t), 'short_name': None}
    for t, f in zip(times_pyin, f0_pyin_v)
]
runs_pyin = collapse_runs(seq_pyin)
print(f'PYIN: {np.sum(voiced_pyin)} voiced frames')
print('First 10 runs:', [(r['short_name'], round(r['end']-r['start'],2)) for r in runs_pyin if r['short_name']][:10])

## A-Step 2: CREPE (already computed)

In [ ]:
# CREPE f0 already in f0_crepe_voiced / time_crepe from Pipeline B
seq_crepe_direct = [
    {**hz_to_swara(float(f), SA_HZ), 'time': float(t)} if not np.isnan(float(f)) else {'time': float(t), 'short_name': None}
    for t, f in zip(time_crepe, f0_crepe_voiced)
]
runs_crepe_direct = collapse_runs(seq_crepe_direct)
print(f'CREPE direct: {np.sum(~np.isnan(f0_crepe_voiced))} voiced frames')
print('First 10 runs:', [(r['short_name'], round(r['end']-r['start'],2)) for r in runs_crepe_direct if r['short_name']][:10])

## A-Step 3: Harmonic Product Spectrum (HPS)

In [ ]:
def hps_pitch(y, sr, hop_length=512, n_fft=4096, n_harmonics=5, fmin=80, fmax=2000):
    stft  = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length))
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
    f_lo, f_hi = np.searchsorted(freqs, fmin), np.searchsorted(freqs, fmax)
    pitches = []
    for frame in stft.T:
        hps = frame.copy()
        for h in range(2, n_harmonics + 1):
            ds = frame[::h][:len(hps)]
            hps[:len(ds)] *= ds
        pitches.append(freqs[np.argmax(hps[f_lo:f_hi]) + f_lo])
    times = librosa.frames_to_time(np.arange(len(pitches)), sr=sr, hop_length=hop_length)
    return times, np.array(pitches)

def energy_voiced_mask(y, sr, hop_length=512, threshold_db=-40):
    rms = librosa.feature.rms(y=y, hop_length=hop_length)[0]
    return librosa.amplitude_to_db(rms, ref=np.max) > threshold_db

times_hps, f0_hps = hps_pitch(y, sr)
voiced_hps = energy_voiced_mask(y, sr)
f0_hps_v   = np.where(voiced_hps[:len(f0_hps)], f0_hps, np.nan)

seq_hps = [
    {**hz_to_swara(float(f), SA_HZ), 'time': float(t)} if not np.isnan(float(f)) else {'time': float(t), 'short_name': None}
    for t, f in zip(times_hps, f0_hps_v)
]
runs_hps = collapse_runs(seq_hps)
print(f'HPS: {np.sum(~np.isnan(f0_hps_v))} voiced frames')
print('First 10 runs:', [(r['short_name'], round(r['end']-r['start'],2)) for r in runs_hps if r['short_name']][:10])

## A-Step 4: Visualise All Direct Methods Side by Side

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

direct_series = [
    ('PYIN (direct)',  times_pyin,  f0_pyin_v,  seq_pyin,  'steelblue'),
    ('CREPE (direct)', time_crepe,  f0_crepe_voiced, seq_crepe_direct, 'tomato'),
    ('HPS (direct)',   times_hps,   f0_hps_v,   seq_hps,   'mediumseagreen'),
]

for ax, (label, times, f0, seq, color) in zip(axes, direct_series):
    semitones = [
        item['semitone'] if item and item.get('semitone') is not None else np.nan
        for item in seq
    ]
    ax.scatter(times, semitones,
               c=[SWARA_COLORS.get(int(s), '#888') if not np.isnan(s) else '#888' for s in semitones],
               s=8, alpha=0.8)
    ax.set_yticks(range(12))
    ax.set_yticklabels([SWARAS_12TET[i][0] for i in range(12)], fontsize=8)
    ax.set_title(f'{label}  (Sa={sa_consensus})')
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(-0.5, 11.5)

axes[-1].set_xlabel('Time (s)')
plt.suptitle('Pipeline A — Direct Flute → Indian Swaras', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

---
## Full Comparison: Pipeline A vs Pipeline B

In [ ]:
# Swara frequency distributions for every method
def swara_distribution(seq_or_events, from_runs=False):
    counts = Counter()
    if from_runs:
        # list of swara event dicts with 'short_name'
        for e in seq_or_events:
            if e and e.get('short_name'):
                dur = (e.get('end', 0) or 0) - (e.get('start', 0) or 0)
                counts[e['short_name']] += max(dur, 0.01)
    else:
        for item in seq_or_events:
            if item and item.get('short_name'):
                counts[item['short_name']] += 1
    return counts


methods = [
    ('A: PYIN',         swara_distribution(seq_pyin)),
    ('A: CREPE',        swara_distribution(seq_crepe_direct)),
    ('A: HPS',          swara_distribution(seq_hps)),
    ('B: Basic-Pitch',  swara_distribution(swaras_from_bp, from_runs=True)),
    ('B: CREPE→MIDI',   swara_distribution(swaras_from_crepe, from_runs=True)),
]
if swaras_from_mt3:
    methods.append(('B: MT3', swara_distribution(swaras_from_mt3, from_runs=True)))

all_swaras = [SWARAS_12TET[i][0] for i in range(12)]

fig, axes = plt.subplots(len(methods), 1, figsize=(12, 3 * len(methods)), sharex=True)

for ax, (label, dist) in zip(axes, methods):
    total = sum(dist.values()) or 1
    fracs = [dist.get(s, 0) / total for s in all_swaras]
    ax.bar(all_swaras, fracs, color=[SWARA_COLORS[i] for i in range(12)], edgecolor='white')
    ax.set_title(label)
    ax.set_ylabel('Fraction')
    ax.set_ylim(0, 0.5)

axes[-1].set_xlabel('Swara')
plt.suptitle(f'Swara Distribution per Method  (Sa={sa_consensus})', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# Agreement matrix: how often does each pair of methods agree on the swara?
import pandas as pd

t_ref = times_pyin   # use PYIN grid as reference

def resample_to_grid(seq, t_ref):
    """Nearest-neighbour resample a frame sequence to a common time grid."""
    times_src = np.array([item.get('time', 0) for item in seq if item])
    semi_src  = np.array([item.get('semitone', np.nan) if item else np.nan for item in seq], dtype=float)
    out = np.full(len(t_ref), np.nan)
    for i, t in enumerate(t_ref):
        if len(times_src) == 0: break
        idx = np.argmin(np.abs(times_src - t))
        out[i] = semi_src[idx]
    return out

def events_to_frame_seq(events, t_ref):
    """Convert note events (with start/end) to a per-frame semitone array."""
    out = np.full(len(t_ref), np.nan)
    for e in events:
        mask = (t_ref >= e['start']) & (t_ref < e['end'])
        out[mask] = e['semitone']
    return out

grids = {
    'A:PYIN':        resample_to_grid(seq_pyin, t_ref),
    'A:CREPE':       resample_to_grid(seq_crepe_direct, t_ref),
    'A:HPS':         resample_to_grid(seq_hps, t_ref),
    'B:BasicPitch':  events_to_frame_seq(swaras_from_bp, t_ref),
    'B:CREPE→MIDI':  events_to_frame_seq(swaras_from_crepe, t_ref),
}
if swaras_from_mt3:
    grids['B:MT3'] = events_to_frame_seq(swaras_from_mt3, t_ref)

# Pairwise frame agreement
keys  = list(grids.keys())
agree = np.zeros((len(keys), len(keys)))
for i, k1 in enumerate(keys):
    for j, k2 in enumerate(keys):
        g1, g2 = grids[k1], grids[k2]
        both_voiced = ~np.isnan(g1) & ~np.isnan(g2)
        if both_voiced.sum() > 0:
            agree[i, j] = np.mean(g1[both_voiced] == g2[both_voiced])

df_agree = pd.DataFrame(agree, index=keys, columns=keys)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(agree, vmin=0, vmax=1, cmap='RdYlGn')
ax.set_xticks(range(len(keys))); ax.set_xticklabels(keys, rotation=40, ha='right')
ax.set_yticks(range(len(keys))); ax.set_yticklabels(keys)
for i in range(len(keys)):
    for j in range(len(keys)):
        ax.text(j, i, f'{agree[i,j]:.2f}', ha='center', va='center', fontsize=9,
                color='black' if agree[i,j] > 0.4 else 'white')
plt.colorbar(im, ax=ax, label='Frame Agreement')
ax.set_title('Pairwise Method Agreement (fraction of frames with same swara)')
plt.tight_layout(); plt.show()
print(df_agree.to_string())

## Gamaka Detection (Ornament / Pitch Glide Regions)

In [ ]:
def detect_gamakas(f0, times, threshold_cents_per_sec=250.0):
    """Flag frames where pitch velocity exceeds threshold (gamaka/ornament)."""
    valid = ~np.isnan(f0)
    cents = np.full_like(f0, np.nan)
    if valid.sum() > 1:
        ref = np.nanmin(f0[valid])
        cents[valid] = 1200 * np.log2(f0[valid] / ref)
    velocity = np.abs(np.gradient(cents, times))
    return (velocity > threshold_cents_per_sec) & valid, velocity

gamaka_mask, pitch_vel = detect_gamakas(f0_pyin_v, times_pyin)

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
axes[0].plot(times_pyin, f0_pyin_v, color='steelblue', lw=1, alpha=0.7, label='PYIN f0')
axes[0].scatter(times_pyin[gamaka_mask], f0_pyin_v[gamaka_mask],
                color='red', s=15, zorder=5, label='Gamaka')
axes[0].set_ylabel('Hz'); axes[0].legend(); axes[0].set_title('Pitch Contour + Gamaka Regions')

axes[1].plot(times_pyin, pitch_vel, color='orange', lw=1)
axes[1].axhline(250, color='red', lw=1.5, linestyle='--', label='threshold')
axes[1].set_ylabel('Cents/sec'); axes[1].set_xlabel('Time (s)')
axes[1].set_title('Pitch Velocity (high = gamaka)')
axes[1].legend()

plt.tight_layout(); plt.show()

n_gamaka = gamaka_mask.sum()
n_voiced = np.sum(~np.isnan(f0_pyin_v))
print(f'Gamaka frames: {n_gamaka} / {n_voiced} voiced ({100*n_gamaka/max(1,n_voiced):.1f}%)')

## Raga Hint from Swara Distribution

In [ ]:
RAGA_TEMPLATES = {
    'Yaman':              [0, 2, 4, 6, 7, 9, 11],
    'Bhairav':            [0, 1, 4, 5, 7, 8, 11],
    'Bhoopali':           [0, 2, 4, 7, 9],
    'Bhimpalasi':         [0, 2, 3, 5, 7, 9, 10],
    'Darbari Kanada':     [0, 2, 3, 5, 7, 8, 10],
    'Malkauns':           [0, 3, 5, 8, 10],
    'Bhairavi':           [0, 1, 3, 5, 7, 8, 10],
    'Todi':               [0, 1, 3, 6, 7, 8, 11],
    'Marwa':              [0, 1, 4, 6, 9, 11],
    'Hansadhwani':        [0, 2, 4, 7, 11],
    'Durga':              [0, 2, 5, 7, 9],
    'Bageshri':           [0, 2, 3, 5, 7, 9, 10],
    'Kedar':              [0, 2, 4, 5, 6, 7, 9, 11],
    'Puriya Dhanashri':   [0, 1, 4, 6, 7, 8, 11],
}

# Use the consensus swara distribution (PYIN direct)
pyin_counts = swara_distribution(seq_pyin)
detected_semitones = set()
for item in seq_pyin:
    if item and item.get('semitone') is not None:
        detected_semitones.add(item['semitone'])

def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if (a | b) else 0

raga_scores = {
    r: jaccard(detected_semitones, s) for r, s in RAGA_TEMPLATES.items()
}
ranked = sorted(raga_scores.items(), key=lambda x: -x[1])

print('Top 5 Raga candidates:')
for raga, score in ranked[:5]:
    swaras = [SWARAS_12TET[i][0] for i in RAGA_TEMPLATES[raga]]
    print(f'  {raga:<25} {score:.3f}  {swaras}')

fig, ax = plt.subplots(figsize=(11, 4))
names_sorted  = [r for r, _ in ranked]
scores_sorted = [s for _, s in ranked]
bars = ax.barh(names_sorted[::-1], scores_sorted[::-1], color='mediumpurple', edgecolor='white')
ax.axvline(0.5, color='red', linestyle='--', alpha=0.5, label='0.5 threshold')
ax.set_xlabel('Jaccard Similarity')
ax.set_title(f'Raga Similarity  (Sa={sa_consensus}, detected via Consensus)')
ax.legend()
plt.tight_layout(); plt.show()

## Export Transcripts

In [ ]:
def export_transcript(runs, filename, sa_name, method_label):
    with open(filename, 'w') as f:
        f.write(f'Method : {method_label}\n')
        f.write(f'Sa     : {sa_name} ({SA_HZ:.2f} Hz)\n')
        f.write('=' * 50 + '\n\n')
        # Inline notation
        line = ''
        for r in runs:
            swara = r.get('short_name')
            if swara:
                dur = (r.get('end') or 0) - (r.get('start') or 0)
                count = max(1, int(round(dur / 0.25)))
                line += ' ' + swara + '-' * (count - 1)
            else:
                line += ' |'
        f.write(line.strip() + '\n\n')
        f.write('Time-stamped events:\n')
        for r in runs:
            if r.get('short_name'):
                dur = (r.get('end') or 0) - (r.get('start') or 0)
                f.write(f"{r.get('start',0):.3f}s – {r.get('end',0):.3f}s  [{dur:.3f}s]  {r['short_name']}\n")
    print(f'Saved → {filename}')

export_transcript(runs_pyin,        'transcript_A_pyin.txt',    sa_consensus, 'Pipeline A — PYIN direct')
export_transcript(runs_crepe_direct,'transcript_A_crepe.txt',   sa_consensus, 'Pipeline A — CREPE direct')
export_transcript(runs_hps,         'transcript_A_hps.txt',     sa_consensus, 'Pipeline A — HPS direct')
export_transcript(collapse_runs(swaras_from_bp), 'transcript_B_basicpitch.txt', sa_consensus, 'Pipeline B — Basic-Pitch → Swaras')
export_transcript(collapse_runs(swaras_from_crepe), 'transcript_B_crepe.txt',   sa_consensus, 'Pipeline B — CREPE→MIDI → Swaras')
if swaras_from_mt3:
    export_transcript(collapse_runs(swaras_from_mt3), 'transcript_B_mt3.txt',   sa_consensus, 'Pipeline B — MT3 → Swaras')

---
## Summary

| Pipeline | Method | Sa Detection | Strengths | Weaknesses |
|----------|--------|-------------|-----------|------------|
| **A** | PYIN | From audio chroma | Fast, voiced/unvoiced decisions | Frame-level noise |
| **A** | CREPE | From audio chroma | High accuracy, viterbi smoothing | GPU helps, slower |
| **A** | HPS | From audio chroma | Pure DSP, no ML | Octave errors on flute |
| **B** | Basic-Pitch → Sa → Swaras | PCP / KS / Low-Freq | Note-level events, MIDI exportable | May miss gamakas |
| **B** | CREPE→MIDI → Sa → Swaras | PCP / KS / Low-Freq | Good pitch accuracy | Needs Sa step correct |
| **B** | **MT3** → Sa → Swaras | PCP / KS / Low-Freq | Best Western transcription, SOTA | Needs JAX + GPU |

### Sa Detection Summary
| Method | Description |
|--------|-------------|
| **PCP Peak Pair** | Finds two most-played notes a perfect fifth apart |
| **Krumhansl-Schmuckler** | Slides scale template over chroma; best-fit = Sa |
| **Low-Freq Isolation** | Bass-only spectrum; dominant pitch = Sa/Pa drone |

**MT3 Setup (one-time)**:
```bash
pip install jax[cuda12] -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
pip install git+https://github.com/magenta/mt3.git
gsutil -q -m cp -r gs://mt3/checkpoints .
```
Then set `MT3_AVAILABLE = True` in the MT3 cell.